In [0]:
%sql
CREATE OR REPLACE TABLE workspace.petropulse_ref.region_production_areas (
  region_id STRING,
  duoarea STRING
);

INSERT INTO workspace.petropulse_ref.region_production_areas VALUES
('R01','STX'),('R01','SNM'),
('R02','STX'),
('R03','SND'),
('R04','SOK'),
('R05','SCO'),
('R06','SPA'),('R06','SWV'),
('R07','R3FM'),
('R08','SAK');

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.petropulse_silver.crude_prices AS
WITH dedup AS (
  SELECT DISTINCT
    CAST(period AS DATE) AS price_date,
    CASE series WHEN 'RWTC' THEN 'WTI' WHEN 'RBRTE' THEN 'Brent' END AS benchmark,
    value AS price_usd_bbl
  FROM workspace.petropulse_bronze.eia_prices_raw
  WHERE series IN ('RWTC','RBRTE')
)
SELECT *,
  LAG(price_usd_bbl) OVER (PARTITION BY benchmark ORDER BY price_date) AS prev_price,
  ROUND((price_usd_bbl / LAG(price_usd_bbl) OVER (PARTITION BY benchmark ORDER BY price_date) - 1) * 100, 2) AS pct_change
FROM dedup;


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.petropulse_silver.regional_production AS
SELECT
  duoarea,
  MAX(area_name) AS area_name,
  CAST(CONCAT(period, '-01') AS DATE) AS month_start,
  MAX(value) AS production_mbbl_d
FROM workspace.petropulse_bronze.eia_production_raw
WHERE process = 'FPF'
  AND units = 'MBBL/D'
  AND (duoarea <> 'SAK' OR series = 'MANFPAK2')
GROUP BY duoarea, CAST(CONCAT(period, '-01') AS DATE);

In [0]:
%sql
SELECT 'crude_prices' AS tbl, COUNT(*) AS n, MIN(price_date) AS first_dt, MAX(price_date) AS last_dt FROM workspace.petropulse_silver.crude_prices
UNION ALL
SELECT 'regional_production', COUNT(*), MIN(month_start), MAX(month_start) FROM workspace.petropulse_silver.regional_production;

In [0]:
%sql
SELECT period, duoarea, series, series_description, value
FROM workspace.petropulse_bronze.eia_production_raw
WHERE duoarea = 'SAK' AND units = 'MBBL/D' AND period = '2026-06';